# Full-spec run on a free T4 (Colab / Kaggle)
All code lives in `.py` files in the repo; this notebook only calls it. Runtime -> **T4 GPU**.
Everything slow-to-make (tokenizer, encoded data, checkpoints) is stored on Drive (Colab) or `/kaggle/working` (Kaggle).

In [ ]:
import os
ON_COLAB = os.path.exists('/content')
if ON_COLAB:
    from google.colab import drive; drive.mount('/content/drive')
    WORK = '/content/drive/MyDrive/minilm_work'
else:
    WORK = '/kaggle/working/minilm_work'
os.makedirs(WORK, exist_ok=True)
REPO_URL = 'https://github.com/<you>/<repo>.git'   # <- put your repo here
!git clone $REPO_URL repo 2>/dev/null || (cd repo && git pull)
%cd repo
!pip -q install -r requirements-dev.txt

In [ ]:
# 1. data (about 2 GB)
D = f'{WORK}/data'; os.makedirs(D, exist_ok=True)
base = 'https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main'
for f in ['TinyStoriesV2-GPT4-valid.txt', 'TinyStoriesV2-GPT4-train.txt']:
    if not os.path.exists(f'{D}/{f}'):
        !wget -q -O {D}/{f} {base}/{f}

In [ ]:
# 2. tokenizer + encoded data (skip if already on Drive). Colab has 2 CPUs: expect ~10-30 min for each step
if not os.path.exists(f'{WORK}/tok/vocab.json'):
    !python scripts/train_tokenizer.py --input {D}/TinyStoriesV2-GPT4-train.txt --out-dir {WORK}/tok --vocab-size 10000 --num-processes 2
for split in ['valid', 'train']:
    if not os.path.exists(f'{WORK}/{split}.npy'):
        !python scripts/encode_dataset.py --input {D}/TinyStoriesV2-GPT4-{split}.txt --out {WORK}/{split}.npy --tokenizer-dir {WORK}/tok --num-processes 2

In [ ]:
# 3. tests + correctness checks (single batch, resume) + learning-rate pilots (5 x 300 steps)
!python -m pytest
!python scripts/run_checks.py --train-data {WORK}/train.npy --val-data {WORK}/valid.npy --runs-dir {WORK}/runs --device cuda
!python scripts/run_pilots.py --train-data {WORK}/train.npy --val-data {WORK}/valid.npy --runs-dir {WORK}/runs --steps 300 --batch-size 32 --lrs 1e-4 3e-4 1e-3 3e-3 1e-2
!python scripts/make_plots.py --runs-dir {WORK}/runs --out-dir report/figures

In [ ]:
# 4. torch.compile speed check, then the main run (5,000 steps x 32 x 256 = 40,960,000 tokens, fp16 + GradScaler)
!python scripts/benchmark_compile.py --steps 100 --out report/compile_benchmark.json
BEST_LR = 3e-3   # <- the best rate from the pilots
!python train.py --train-data {WORK}/train.npy --val-data {WORK}/valid.npy --out-dir {WORK}/runs/main --lr {BEST_LR} --max-steps 5000 --batch-size 32 --precision fp16 --compile --eval-interval 250 --ckpt-interval 250
# if the session disconnects: re-run this cell with --resume appended

In [ ]:
# 5. package + stories
!python scripts/make_release.py --checkpoint {WORK}/runs/main/final_model.pt --tokenizer-dir {WORK}/tok --release-dir release
!python scripts/sample_stories.py --model-dir release --out report/stories.md
!python inference.py --model-dir release --prompt "Once upon a time" --max-new-tokens 256 --temperature 0.8 --top-p 0.9 --seed 42 --device cpu